# Hallucination and grounding

Exercise notebook for the lesson **[Hallucination and grounding](https://ramkumarmd.com/medical-ai/generative-ai-llms/hallucination-and-grounding)** in the *Generative AI & LLMs* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** In clinical text, a hallucination is content that is not supported by the source, such as a medicine the patient was never prescribed; an omission is something important that was left out. Both occur even in good systems: one large evaluation of AI-generated clinical notes found hallucinations in 1.47% of sentences, almost half of them potentially harmful. Grounding the model in specific source documents, requiring traceable citations and checking outputs against the source reduce the risk, but a clinician must still review what matters.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Measuring errors with clinician review

Summarising a clinician review of four synthetic AI-generated notes.

In [ ]:
reviews = [
    {"note_sentences": 14, "hallucinations": [], "omissions": ["allergy"]},
    {"note_sentences": 11, "hallucinations": [("dose changed", "major")], "omissions": []},
    {"note_sentences": 16, "hallucinations": [("extra symptom", "minor")],
     "omissions": ["follow-up date", "smoking status"]},
    {"note_sentences": 12, "hallucinations": [], "omissions": []},
]
sentences = sum(r["note_sentences"] for r in reviews)
halluc = [h for r in reviews for h in r["hallucinations"]]
omissions = sum(len(r["omissions"]) for r in reviews)
major = sum(1 for _, severity in halluc if severity == "major")
print(f"{sentences} sentences reviewed")
print(f"Hallucination rate {len(halluc) / sentences:.1%} ({major} major)")
print(f"Omissions: {omissions} across {len(reviews)} notes")
print(f"Notes with at least one error: "
      f"{sum(1 for r in reviews if r['hallucinations'] or r['omissions'])} of {len(reviews)}")

# 53 sentences reviewed
# Hallucination rate 3.8% (1 major)
# Omissions: 3 across 4 notes
# Notes with at least one error: 3 of 4

### A simple automated check, and its limits

Flag summary sentences containing numbers or medicines that are not in the synthetic source.

In [ ]:
import re

source = """Synthetic consultation. Patient reports tiredness for three weeks.
On metformin 1 g twice daily. HbA1c last month 8.4%. Blood pressure today 138/86.
No chest pain. Plan: add a second medicine after discussing options; recheck HbA1c in 3 months."""

summary_sentences = [
    "Three weeks of tiredness.",
    "Takes metformin 1 g twice daily.",
    "HbA1c 8.4% last month; blood pressure 138/86.",
    "Started on gliclazide 80 mg.",          # not in the source
    "Reports chest pain on exertion.",       # contradicts the source
    "Recheck HbA1c in 6 months.",            # wrong interval
]

MEDICINES = {"metformin", "gliclazide", "insulin", "sitagliptin", "empagliflozin"}
src = source.lower()
src_numbers = set(re.findall(r"\d+(?:\.\d+)?", src))

def unsupported_details(sentence):
    s = sentence.lower()
    problems = [f"number {n}" for n in re.findall(r"\d+(?:\.\d+)?", s) if n not in src_numbers]
    problems += [f"medicine {m}" for m in MEDICINES if m in s and m not in src]
    return problems

for sentence in summary_sentences:
    problems = unsupported_details(sentence)
    print(("FLAG  " if problems else "ok    ") + sentence, problems or "")

# ok    Three weeks of tiredness.
# ok    Takes metformin 1 g twice daily.
# ok    HbA1c 8.4% last month; blood pressure 138/86.
# FLAG  Started on gliclazide 80 mg. ['number 80', 'medicine gliclazide']
# ok    Reports chest pain on exertion.
# FLAG  Recheck HbA1c in 6 months. ['number 6']

## Exercise data

Synthetic data written for this exercise (no real patients). Run this cell before starting the steps.

In [ ]:
# Ten synthetic consultation transcripts (no real patients), each with an AI-style draft note written to
# contain the kinds of errors seen in AI documentation, and clinician-style labels.
# Label types: "hallucination" (not supported by the transcript), "contradiction" (says the opposite of the
# transcript). Omissions are listed separately. Severity: "major" could change care; "minor" would not.
cases = [
    {"transcript": "Doctor: How have your sugars been? Patient: Mostly 140 to 180 fasting. Doctor: Any low sugars? "
                   "Patient: No. Doctor: You're on metformin 500 mg twice daily. HbA1c is 8.2%. Let's increase metformin "
                   "to 1 g twice daily. Patient: I'm allergic to penicillin, by the way. Doctor: Noted. Review in 3 months.",
     "draft": ["Fasting glucose 140-180 mg/dL.", "No hypoglycaemia.", "On metformin 500 mg twice daily.",
               "HbA1c 8.2%.", "Metformin increased to 1 g twice daily.", "Review in 3 months."],
     "errors": [], "omissions": [("penicillin allergy", "major")]},
    {"transcript": "Patient: My feet tingle at night. Doctor: Any wounds? Patient: No. Doctor: Monofilament reduced on both "
                   "feet. HbA1c 7.6%. I'll refer you to podiatry. Blood pressure 134/82.",
     "draft": ["Night-time tingling in both feet.", "Small ulcer on the left heel.", "Reduced monofilament sensation bilaterally.",
               "HbA1c 7.6%; blood pressure 134/82.", "Referred to podiatry."],
     "errors": [(1, "contradiction", "major")], "omissions": []},
    {"transcript": "Doctor: Your kidney test shows eGFR 48. We'll start empagliflozin 10 mg daily. Any urinary symptoms? "
                   "Patient: No. Doctor: Recheck kidney function in 4 weeks.",
     "draft": ["eGFR 48.", "Started empagliflozin 25 mg daily.", "No urinary symptoms.", "Kidney function recheck in 4 weeks."],
     "errors": [(1, "hallucination", "major")], "omissions": []},
    {"transcript": "Patient: I've been more tired lately. Doctor: Any chest pain or breathlessness? Patient: No chest pain. "
                   "Doctor: Haemoglobin 10.9. Let's check iron studies and B12. You're on metformin 1 g twice daily.",
     "draft": ["Tiredness recently.", "Reports chest pain on exertion.", "Haemoglobin 10.9.",
               "Iron studies and B12 requested.", "On metformin 1 g twice daily."],
     "errors": [(1, "contradiction", "major")], "omissions": []},
    {"transcript": "Doctor: Your HbA1c is 6.4%. You're on gliclazide 80 mg. Any dizzy spells? Patient: Yes, twice last week, "
                   "before lunch. Doctor: Let's reduce gliclazide to 40 mg. Patient: OK.",
     "draft": ["HbA1c 6.4%.", "Two episodes of dizziness before lunch last week, likely hypoglycaemia.",
               "Gliclazide reduced to 40 mg.", "Advised to stop driving for 6 weeks."],
     "errors": [(3, "hallucination", "major")], "omissions": []},
    {"transcript": "Doctor: Blood pressure today 148/92. You take ramipril 5 mg. Patient: I sometimes forget it. Doctor: Let's "
                   "use a pill organiser and recheck in 2 weeks. Your cholesterol LDL is 2.1.",
     "draft": ["Blood pressure 148/92 on ramipril 5 mg.", "Occasionally forgets doses.", "Pill organiser suggested.",
               "LDL 2.1.", "Smoker, 10 cigarettes a day.", "Blood pressure recheck in 2 weeks."],
     "errors": [(4, "hallucination", "minor")], "omissions": []},
    {"transcript": "Patient: I'm 30 weeks pregnant. Doctor: Your fasting readings are 5.6 to 6.0. We'll start insulin at "
                   "night, 6 units. Any headaches or swelling? Patient: Some ankle swelling. Doctor: Blood pressure 128/80.",
     "draft": ["30 weeks pregnant.", "Fasting glucose 5.6-6.0 mmol/L.", "Night-time insulin started, 6 units.",
               "No swelling.", "Blood pressure 128/80."],
     "errors": [(3, "contradiction", "minor")], "omissions": []},
    {"transcript": "Doctor: You've lost 4 kg since starting semaglutide. Any nausea? Patient: A little in the first week, "
                   "gone now. Doctor: HbA1c 7.1%. Continue the same dose. Review in 3 months.",
     "draft": ["Weight down 4 kg on semaglutide.", "Mild nausea in the first week, now resolved.", "HbA1c 7.1%.",
               "Dose increased.", "Review in 3 months."],
     "errors": [(3, "contradiction", "major")], "omissions": [("continue same dose", "major")]},
    {"transcript": "Patient: My vision is blurry sometimes. Doctor: Your last eye screen was 2 years ago. I'll book retinal "
                   "screening. HbA1c 9.4%. Doctor: We'll add a second medicine; let's discuss options next visit.",
     "draft": ["Intermittent blurred vision.", "Last eye screening 2 years ago; retinal screening booked.",
               "HbA1c 9.4%.", "Started sitagliptin 100 mg daily.", "Shortness of breath on stairs."],
     "errors": [(3, "hallucination", "major"), (4, "hallucination", "minor")], "omissions": []},
    {"transcript": "Doctor: Any foot problems? Patient: No. Doctor: Pulses present, sensation normal. HbA1c 6.9%. "
                   "Blood pressure 126/78. Keep going as you are. Review in 6 months.",
     "draft": ["No foot problems.", "Foot pulses present, sensation normal.", "HbA1c 6.9%.",
               "Blood pressure 126/78.", "Review in 6 months."],
     "errors": [], "omissions": []},
]
print(f"{len(cases)} cases, {sum(len(c['draft']) for c in cases)} draft sentences")

## Exercise

**Goal:** Measure and reduce hallucinations and omissions in synthetic AI-generated clinic notes.

**Data:** Ten synthetic consultation transcripts with AI-style draft notes, written for the notebook (Created for this course; no real patient data)

### Step 1

Review two draft notes sentence by sentence against their transcripts and label hallucinations, contradictions and omissions with severity.

In [ ]:
# Your code here


### Step 2

Compute per-sentence and per-note error rates for all ten notes using the labels provided.

In [ ]:
# Your code here


### Step 3

Run the number-and-medicine checker and calculate its sensitivity for the labelled hallucinations.

In [ ]:
# Your code here


### Step 4

Add a simple negation rule (for example 'no', 'denies', 'without' before a symptom) and measure the improvement.

In [ ]:
# Your code here


### Step 5

Write a one-page review protocol proportionate to how these notes would be used.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/generative-ai-llms/hallucination-and-grounding](https://ramkumarmd.com/medical-ai/generative-ai-llms/hallucination-and-grounding). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.